# Testing vortexmaps

Verifies that `vortexmap.py` reproduces the **inline** implementation that
`Preloaded Vortexmap.ipynb` has been using all along (TODO 1f-7, 1f-8).

Run top to bottom. Cells 1-3 read a single file and print what they find, so
stop there first if anything looks wrong.

**Scope:** OSAR bootstrap data only. The climbing folder holds
`*_deltag_allstats.csv` (wide format, no `Metric` / `Light_Intensity` columns), which
`load_bootstrap_data` cannot read. Climbing returns here after file regeneration.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import NLProcessing
import vortexmap

In [ ]:
laptop = "C:\\Users\\lnico\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"
homecomp = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"   # TODO: update when home Dropbox moves to NUS Dropbox
labcomp = "C:\\Users\\User\\NUS Dropbox\\acclab\\Nicole M Lee"
specifiedpath = labcomp

osardir = "\\Data Compilation\\osar_compiled\\Bootstrapped stats\\"
osarpath = specifiedpath + osardir

## 1. Schema check - reads ONE file

In [ ]:
sample = pd.read_csv(osarpath + "MB011B x Chrimson2_bootstrap.csv")
print(sample.columns.tolist())
print(sorted(sample["Metric"].unique()))
print(sorted(sample["Light_Intensity"].unique()))

## 2. Reference implementation

Verbatim copy of `Preloaded Vortexmap.ipynb` cell 2. This is the baseline.

In [ ]:
# --- reference copies, do not edit: verbatim from Preloaded Vortexmap.ipynb cell 2 ---
def _sample_bootstrap(bootstrap, m, n, reverse_neg, abs_rank, chop_tail):

    bootstrap_sorted = sorted(bootstrap)
    chop_tail_int = int(np.ceil(len(bootstrap_sorted) * chop_tail / 100))
    bootstrap_sorted = bootstrap_sorted[chop_tail_int : len(bootstrap_sorted) - chop_tail_int]

    ranks_to_look = np.linspace(0, len(bootstrap_sorted), m * n, dtype=int)
    ranks_to_look[0] = 1

    if np.sum(np.array(bootstrap_sorted) > 0) < len(bootstrap_sorted) / 2:
        if reverse_neg:
            bootstrap_sorted = bootstrap_sorted[::-1]

    if abs_rank:
        bootstrap_sorted = sorted(bootstrap_sorted, key=abs)

    long_ranks = [bootstrap_sorted[r - 1] for r in ranks_to_look]
    return long_ranks


def _spiralize(fill, m, n):

    i = 0
    j = 0
    k = 0
    array = np.zeros((m, n))

    while m > 0 and k < len(fill):
        jj = j
        ii = i

        # Right
        for j in range(j, n):
            if k >= len(fill):
                break
            array[i, j] = fill[k]
            k += 1

        # Down
        for i in range(ii + 1, m):
            if k >= len(fill):
                break
            array[i, j] = fill[k]
            k += 1

        # Left
        for j in range(n - 2, jj - 1, -1):
            if k >= len(fill):
                break
            array[i, j] = fill[k]
            k += 1

        # Up
        for i in range(m - 2, ii, -1):
            if k >= len(fill):
                break
            array[i, j] = fill[k]
            k += 1

        m -= 1
        n -= 1
        j += 1

    return array

## 3. Load via the module

In [ ]:
METRICS = sorted(sample["Metric"].unique())
RESPONDER = "Chrimson2"

osardata = vortexmap.load_bootstrap_data(osarpath, METRICS, responders=[RESPONDER])
print("responders:", list(osardata))
print("metrics:", len(METRICS), "| MBONs:", len(osardata[RESPONDER][METRICS[0]]))

## 4. Equivalence: module vs reference

`_sample_bootstrap` and `_spiralize` should agree exactly. Note the reference takes
`(bootstrap, m, n, reverse_neg, abs_rank, chop_tail)` while the module takes
`(bootstrap, m, n, chop_tail, reverse_neg, abs_rank)` - the order differs, so both
calls below pass by keyword.

In [ ]:
N = 11
CHOP = 2.5
mismatch = []

for metric in METRICS:
    for mbon in sorted(osardata[RESPONDER][metric]):
        bs = vortexmap.get_bootstrap(osardata, RESPONDER, metric, mbon)

        ref_ranks = _sample_bootstrap(bs, N, N, reverse_neg=True, abs_rank=False, chop_tail=CHOP)
        mod_ranks = vortexmap._sample_bootstrap(bs, N, N, chop_tail=CHOP, reverse_neg=True, abs_rank=False)

        ref_spiral = _spiralize(ref_ranks, N, N)
        mod_spiral = vortexmap._spiralize(mod_ranks, N, N)

        if not np.allclose(ref_ranks, mod_ranks) or not np.allclose(ref_spiral, mod_spiral):
            mismatch.append((metric, mbon))

print(f"checked {len(METRICS)} metrics x {len(osardata[RESPONDER][METRICS[0]])} MBONs")
print("mismatches:", len(mismatch))
if mismatch:
    print(mismatch[:10])

## 5. Centre annotation - the one deliberate difference

Reference annotates `np.mean(long_ranks)`; the module annotates the observed
Hedges' g from the CSV (TODO 1f-7 item 1). They are **expected to differ** - this
cell quantifies by how much, it is not a failure.

In [ ]:
rows = []
for metric in METRICS:
    for mbon in sorted(osardata[RESPONDER][metric]):
        bs = vortexmap.get_bootstrap(osardata, RESPONDER, metric, mbon)
        ranks = vortexmap._sample_bootstrap(bs, N, N, chop_tail=CHOP)
        effect, _, _ = vortexmap.get_effect(osardata, RESPONDER, metric, mbon)
        rows.append({"metric": metric, "mbon": mbon,
                     "mean_long_ranks": np.mean(ranks), "hedges_g": effect})

cmp = pd.DataFrame(rows)
cmp["diff"] = cmp["mean_long_ranks"] - cmp["hedges_g"]
print(cmp["diff"].describe())
cmp.head(10)

## 6. MBON ordering and labels

Exercises `resolve_order` / `create_labels` / `create_mbon_only_labels` - the paths the
smoke test never touched.

In [ ]:
mbons = sorted(osardata[RESPONDER][METRICS[0]])
lobelocation = NLProcessing.generate_lobelocation(mbons, "MBONlist.csv")

print("alphabetical :", vortexmap.resolve_order(osardata, RESPONDER, lobelocation)[:6])
print("by number    :", vortexmap.resolve_order(osardata, RESPONDER, lobelocation, sort_by="mbon_number")[:6])
print("y labels     :", vortexmap.create_labels(mbons, lobelocation, sep=" | ")[:3])
print("x labels     :", vortexmap.create_mbon_only_labels(mbons, lobelocation)[:6])

## 7. Draw - eyeball `_get_text_color` (TODO 1f-6)

Check the annotation text stays readable against both dark and light cells.

In [ ]:
spirals, mean_vals, mbons = vortexmap.build_vortex_df(
    osardata, RESPONDER, METRICS, lobelocation, n=N, chop_tail=CHOP)

fig, ax = vortexmap.vortex_map(spirals, mean_vals, n=N, cmap="coolwarm",
                               vmin=-1.5, vmax=1.5, figsize=(6, 18),
                               annot=True, annot_fontsize=9)
ax.set_yticklabels(vortexmap.create_labels(mbons, lobelocation, sep=chr(10)), fontsize=8)
ax.set_xticklabels(METRICS, fontsize=9, rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. One-tone check

Single-hue colourmap with `center=None` - text colour must still adapt.

In [ ]:
fig, ax = vortexmap.vortex_map(spirals, mean_vals, n=N, figsize=(6, 18),
                               heatmap_kwargs={"cmap": "rocket", "center": None})
ax.set_yticklabels(vortexmap.create_labels(mbons, lobelocation, sep=chr(10)), fontsize=8)
ax.set_xticklabels(METRICS, fontsize=9, rotation=45, ha="right")
plt.tight_layout()
plt.show()